# Analyse complète bout-en-bout — Énergie allemande (ENTSO-E)

**Mini-projet Séries Temporelles (TEK-UP).** Séries horaires `Load`, `Wind`,
`Solar` (MW), Allemagne, `2015-01-01 → 2020-09-30` (50 400 h UTC), découpage
**chronologique 90 / 10** (test à partir du 05/03/2020, 5 040 h).

Ce notebook suit tout le pipeline :
1. Données & EDA · 2. Stationnarité · 3. ARIMA/SARIMA · 4. GARCH ·
5. VAR/VECM · 6. LSTM/GRU · 7. Prophet & hybride · 8. Évaluation comparée ·
9. Volet IA générative.

> **Exécution.** Toutes les cellules lisent les résultats déjà produits dans
> `results/` (léger). Pour **régénérer** ces résultats, exécuter les scripts
> R (`R/01…06`) puis Python dans l'ordre — cellule « Régénération du
> pipeline » en annexe (`RUN_PIPELINE = True`).

In [ ]:
# --- Environnement ---
import sys, subprocess, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PY = ROOT / "Python"
TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"
FORE = ROOT / "results" / "forecasts"
sys.path.insert(0, str(PY))
print("Racine :", ROOT)

RUN_PIPELINE = False   # passer à True pour tout regénérer (long, gourmand RAM)

## 1. Données et statistiques descriptives

In [ ]:
from data_preparation import load_germany_series
from arima_var import na_interp_hour
VARIABLES = ["Load", "Wind", "Solar"]

df = load_germany_series().copy()
for v in VARIABLES:
    df[v] = na_interp_hour(df[v])
print(df.shape, df.index.min(), "->", df.index.max())
df[VARIABLES].describe().round(1)

In [ ]:
display(pd.read_csv(TAB / "missing_values.csv")) if (TAB / "missing_values.csv").exists() else None
display(pd.read_csv(TAB / "data_overview_summary.csv")) if (TAB / "data_overview_summary.csv").exists() else None

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for ax, v in zip(axes, VARIABLES):
    ax.plot(df.index, df[v], lw=0.4, color={"Load": "#2563eb", "Wind": "#10b981", "Solar": "#f59e0b"}[v])
    ax.set_ylabel(v + " (MW)"); ax.set_title(v)
plt.tight_layout(); plt.show()

## 2. Analyse exploratoire (profils horaire / hebdo / mensuel, saisonnalité)

In [ ]:
for f in ["hourly_summary.csv", "weekly_summary.csv", "monthly_summary.csv", "yearly_summary.csv"]:
    p = TAB / f
    if p.exists():
        print("=====", f); display(pd.read_csv(p).round(1))

In [ ]:
try:
    from statsmodels.tsa.seasonal import STL
    s = df["Load"].asfreq("h").interpolate()
    res = STL(s[-24*14:], period=24).fit()
    res.plot(); plt.suptitle("STL - Load (2 semaines, période 24h)"); plt.tight_layout(); plt.show()
except Exception as e:
    print("STL ignoré :", e)

## 3. Stationnarité (ADF / KPSS, R et Python)

In [ ]:
for f in ["stationarity_tests.csv", "R_stationarity_tests.csv"]:
    p = TAB / f
    if p.exists():
        print("=====", f); display(pd.read_csv(p))

## 4. ARIMA / SARIMA

Trois candidats par série : `SARIMA_manuel` (1,1,1)(1,1,1)[24], `ARIMA_auto`
(ordre auto, non saisonnier) et `SARIMA_auto` (auto.arima R). Sélection par
**AICc** minimal.

In [ ]:
comp = pd.read_csv(TAB / "py_arima_model_comparison.csv")
R_comp = pd.read_csv(TAB / "R_arima_model_comparison.csv") if (TAB / "R_arima_model_comparison.csv").exists() else None
display(comp.round(1))
if R_comp is not None: display(R_comp.round(1))
best = comp.loc[comp.groupby("Variable")["AICc"].idxmin()]
print("Meilleur AICc par variable :"); display(best[["Variable", "Modele", "Spec", "AICc"]])

In [ ]:
met = pd.read_csv(TAB / "py_arima_metrics.csv")
display(met.round(3))
for v in VARIABLES:
    for h in sorted(met.Horizon.unique()):
        sub = met[(met.Variable == v) & (met.Horizon == h)]
        b = sub.loc[sub.RMSE.idxmin()]
        print(f"{v} h={h}: meilleur {b.Modele} (RMSE={b.RMSE:.1f})")

In [ ]:
from IPython.display import Image, display as disp
for v in VARIABLES:
    p = FIG / f"py_prev_{v}.png"
    if p.exists():
        print(v); disp(Image(filename=str(p)))

In [ ]:
p = TAB / "py_arima_intervals.csv"
if p.exists():
    it = pd.read_csv(p)
    display(it[it.Horizon <= 24].head(12))
    print("Bornes renseignées :", it.dropna(subset=["lower"]).shape[0], "lignes avec IC")
else:
    print("Intervalles non encore produits (lancer arima_var.py / arima_intervals.py).")

## 5. Volatilité : GARCH / EGARCH

In [ ]:
for f in ["py_garch_coefficients.csv", "py_garch_persistence.csv", "py_garch_diagnostics.csv"]:
    p = TAB / f
    if p.exists():
        print("=====", f); display(pd.read_csv(p))

In [ ]:
for v in VARIABLES:
    p = FIG / f"py_garch_{v}.png"
    if p.exists():
        print(v); disp(Image(filename=str(p)))

## 6. VAR / Granger / Johansen / VECM (données journalières)

In [ ]:
for f in ["py_var_order.csv", "py_granger.csv", "py_johansen.csv", "py_vecm.csv", "py_vecm_alpha.csv", "py_var_fevd.csv"]:
    p = TAB / f
    if p.exists():
        print("=====", f); display(pd.read_csv(p))

## 7. Deep learning : LSTM / GRU (Dropout, validation croisée temporelle)

In [ ]:
p = TAB / "py_lstm_gru_metrics.csv"
if p.exists():
    display(pd.read_csv(p).round(3))
for v in VARIABLES:
    for kind in ["forecast", "loss"]:
        fp = FIG / f"py_lstm_gru_{kind}_{v}.png"
        if fp.exists():
            print(f"{kind} {v}"); disp(Image(filename=str(fp)))

In [ ]:
p = TAB / "py_lstm_gru_cv.csv"
if p.exists():
    display(pd.read_csv(p).round(2))
    cv = pd.read_csv(p)
    try:
        cv.groupby(["Variable", "Modele"])["RMSE"].agg(["mean", "std"]).round(2).plot.bar(y="mean", ylabel="RMSE moyen (TSCV)")
        plt.tight_layout(); plt.show()
    except Exception as e:
        print(e)
else:
    print("Validation croisée temporelle non encore produite (lancer lstm_gru.py).")

## 8. Modèles additionnels : Prophet et hybride SARIMA+LSTM

In [ ]:
for f in ["py_prophet_metrics.csv", "py_hybrid_metrics.csv"]:
    p = TAB / f
    if p.exists():
        print("=====", f); display(pd.read_csv(p).round(3))
    else:
        print("(absent)", f)
for v in VARIABLES:
    for pref in ["py_prophet_", "py_hybrid_"]:
        fp = FIG / f"{pref}{v}.png"
        if fp.exists():
            print(fp.name); disp(Image(filename=str(fp)))

## 9. Évaluation comparée (leaderboard, walk-forward, Diebold–Mariano)

In [ ]:
rank = pd.read_csv(TAB / "py_evaluation_ranking.csv")
print("Classement RMSE (top 15) :")
display(rank.sort_values("RMSE").head(15).round(2))
try:
    pivot = rank.pivot_table(index="Modele", columns="Horizon", values="RMSE", aggfunc="min")
    display(pivot.round(0))
except Exception as e:
    print(e)

In [ ]:
wf = pd.read_csv(TAB / "py_walkforward_metrics.csv")
display(wf.round(2))
dm = pd.read_csv(TAB / "py_diebold_mariano.csv")
display(dm)
if (TAB / "py_model_comparison.csv").exists():
    display(pd.read_csv(TAB / "py_model_comparison.csv"))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, v in zip(axes, VARIABLES):
    fp = FORE / f"py_walkforward_{v}.csv"
    if not fp.exists():
        continue
    d = pd.read_csv(fp, parse_dates=["DateTime"]).tail(24 * 14)
    ax.plot(d["DateTime"], d["Observe"], color="black", lw=1, label="Observé")
    for c in d.columns:
        if c not in ("Observe", "DateTime"):
            ax.plot(d["DateTime"], d[c], lw=1, label=c)
    ax.set_title(f"Walk-forward 1 pas - {v} (fenêtre 2 semaines)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
p = FIG / "py_evaluation_rmse.png"
if p.exists(): disp(Image(filename=str(p)))

## 10. Volet IA générative (protocole manuel, sans API)

In [ ]:
import glob
llm = sorted(glob.glob(str(ROOT / "results" / "llm" / "echanges" / "*.md")))
print("Échanges archivés :")
for f in llm: print(" -", Path(f).name)
if llm:
    print("\n===== Extrait du dernier échange =====\n")
    print(Path(llm[-1]).read_text(encoding="utf-8")[:1800])

## 11. Conclusion

- **Court terme (h=10/24)** : SARIMA saisonnier et LSTM proches sur Load ;
  ARIMA/SARIMA robustes sur Wind.
- **Long terme (h=5040)** : le LSTM récursif limite la dérive sur Load ;
  dégradation générale, MAPE inexploitable sur Solar (zéros nocturnes).
- **Walk-forward 1 pas** : LSTM/GRU bat significativement les baselines
  naïves (Diebold–Mariano, p < 1e-60).
- **Limites** : période de test au début du COVID, pas de régresseurs
  saisonniers externes (Fourier, m=168), Solar intermittente.

## Annexe — Régénération du pipeline

Passer `RUN_PIPELINE = True` dans la première cellule pour tout recalculer
(long : SARIMA saisonnier ~7 min/série, réseaux de neurones, etc.).

In [ ]:
if RUN_PIPELINE:
    scripts = ["data_preparation.py", "arima_var.py", "arima_intervals.py",
               "garch.py", "lstm_gru.py", "prophet_model.py",
               "hybrid_arima_lstm.py", "evaluation.py"]
    for s in scripts:
        sp = PY / s
        if sp.exists():
            print(">>>", s, flush=True)
            subprocess.run([sys.executable, str(sp)], check=False)
else:
    print("RUN_PIPELINE=False : cellules en lecture seule des résultats existants.")